# Google Drive BirdNET Analyzer

This notebook created by [biodiversica](https://biodiversica.xyz) reads audio recordings directly from your **Google Drive** and runs **BirdNET v2.4** to detect bird species in each recording.

---

### What this notebook does (in order):
1. **Connects to your Google Drive** to read audio and save results
2. **Installs the necessary software** automatically
3. **Scans your audio folder** to discover recording sites and files
4. **Loads BirdNET** and builds an optional location-based species filter
5. **Analyzes every recording** and saves the results as CSV files on your Drive

### Folder structure:
You can organize your audio files in two ways:

```
# Flat — all files in one folder (single site)
MyDrive/audio/
  20250101_180000.wav
  20250101_183000.wav

# Subfolders — each subfolder is a separate recording site
MyDrive/audio/
  POINT_A/
    20250101_180000.wav
  POINT_B/
    20250101_180000.wav
```

### Filename format:
The notebook reads timestamps from filenames. Supported formats:
- **AudioMoth** (default): `YYYYMMDD_HHMMSS.wav` — e.g. `20250615_203000.wav`
- **AudioMoth with prefix**: `PREFIX_YYYYMMDD_HHMMSS.wav` — e.g. `SM4_20250615_203000.wav`
- **ISO datetime**: `YYYY-MM-DD_HH-MM-SS.wav` — e.g. `2025-06-15_20-30-00.wav`

Files whose names do not match any pattern are still analyzed — the timestamp-based columns in results will be left blank.

### Before you start:
- You need a **Google account** with Google Drive
- Your audio files must be on your Google Drive
- The **latitude and longitude** of your recording sites are recommended for the location filter

### How to run:
Run each cell **one at a time**, from top to bottom. Click the ▶ button on the left of each cell, or press `Shift + Enter`.

> **New to notebooks?** A cell with `[ ]` on the left has not been run. After running, it shows `[1]`, `[2]`, etc. If you see an error (red text), read the message — it usually tells you exactly what to fix.

---
## Step 1 — Connect Google Drive & Install Software

Run the two cells below. The first will ask you to **allow access to your Google Drive** — click the link and follow the steps.

In [ ]:
#@title 📂 Connect Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive connected successfully.')

In [ ]:
#@title 📦 Install packages { display-mode: "form" }

!pip install birdnet -q
!apt-get install -y -q libsndfile1

print('\nAll packages installed successfully.')

---
## Step 2 — Configuration

Fill in the forms below. **You do not need to edit any code** — just type or select your values in each form and run the cell.

Run all three forms from top to bottom:
1. **General Settings** — confidence threshold, results folder
2. **Audio Input** — where your recordings are, how filenames are formatted, and an optional date/time filter
3. **BirdNET Settings** — location filter and model options

> **Tip:** The forms hide the code automatically. To see the underlying code, click the `{ }` icon in the top-right corner of any form cell.

In [ ]:
#@title ⚙️ General Settings { display-mode: "form" }

import os

#@markdown **Results folder** — where detection results will be saved on your Google Drive.
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/audio_birdnet_results"  #@param {type:"string"}

#@markdown ---
#@markdown **Detection threshold** — minimum confidence score to save a detection (0.0–1.0).
#@markdown Lower = more detections (may include false positives). Higher = fewer but more reliable.
MIN_CONFIDENCE = 0.25  #@param {type:"slider", min:0.0, max:1.0, step:0.05}

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"Results folder  : {DRIVE_RESULTS_DIR}")
print(f"Min. confidence : {MIN_CONFIDENCE}")

In [ ]:
#@title ⚙️ Audio Preprocessing { display-mode: "form" }
#@markdown Optional preprocessing applied to each recording before inference.
#@markdown
#@markdown ---
#@markdown **Frequency filter** — remove frequencies outside the range of interest.
FILTER_TYPE = "none"  #@param ["none", "lowpass", "highpass", "bandpass"]

#@markdown **Low-cut frequency (Hz)** — used for highpass and bandpass filters.
FILTER_LOW_HZ = 0  #@param {type:"integer"}

#@markdown **High-cut frequency (Hz)** — used for lowpass and bandpass filters.
FILTER_HIGH_HZ = 15000  #@param {type:"integer"}

#@markdown ---
#@markdown **Playback speed** — 1.0 = normal. Below 1.0 slows down and lengthens audio;
#@markdown above 1.0 speeds up and shortens it. Useful for shifting frequency content
#@markdown into the model's expected range (e.g. 0.5x halves all frequencies).
AUDIO_SPEED = 1.0  #@param {type:"slider", min:0.25, max:4.0, step:0.05}

_preprocess_lines = []
if FILTER_TYPE != 'none':
    if FILTER_TYPE == 'lowpass':
        _preprocess_lines.append(f'Filter   : lowpass <= {FILTER_HIGH_HZ} Hz')
    elif FILTER_TYPE == 'highpass':
        _preprocess_lines.append(f'Filter   : highpass >= {FILTER_LOW_HZ} Hz')
    elif FILTER_TYPE == 'bandpass':
        _preprocess_lines.append(f'Filter   : bandpass {FILTER_LOW_HZ}-{FILTER_HIGH_HZ} Hz')
if AUDIO_SPEED != 1.0:
    _preprocess_lines.append(f'Speed    : {AUDIO_SPEED}x')
if _preprocess_lines:
    print('Preprocessing enabled:')
    for _l in _preprocess_lines:
        print(f'  {_l}')
else:
    print('Preprocessing : none')

#@markdown ---
#@markdown **Time of day filter** — only analyze recordings within a specific daily time window.
#@markdown Leave disabled to analyze all recordings regardless of time.
TIME_FILTER_ENABLED = False  #@param {type:"boolean"}
#@markdown **Window start (HH:MM)**
TIME_FILTER_START = "06:00"  #@param {type:"string"}
#@markdown **Window end (HH:MM)** — can cross midnight, e.g. 22:00 to 06:00.
TIME_FILTER_END = "20:00"  #@param {type:"string"}

if TIME_FILTER_ENABLED:
    print(f'Time filter : {TIME_FILTER_START} – {TIME_FILTER_END}')

In [ ]:
#@title 🗂️ Audio Input { display-mode: "form" }

#@markdown **Audio folder** — path to the folder on your Google Drive that contains the recordings.
#@markdown - If the folder has **subfolders**, each subfolder will be treated as a separate recording site.
#@markdown - If all files are directly in this folder, the folder name is used as the site name.
#@markdown Example: `/content/drive/MyDrive/my_project/audio`
DRIVE_INPUT_DIR = "/content/drive/MyDrive/audio"  #@param {type:"string"}

#@markdown ---
#@markdown **Filename prefix** — leave blank for standard AudioMoth filenames (`YYYYMMDD_HHMMSS.wav`).
#@markdown If your recorder adds a prefix before the timestamp, enter it here **without** the trailing underscore.
#@markdown Example: enter `SM4` for files named `SM4_20250615_203000.wav`
FILENAME_PREFIX = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Date/time filter** (optional) — when enabled, only files whose filename timestamp
#@markdown falls within this range will be analyzed. Files with no parseable timestamp are always included.
#@markdown Leave disabled to analyze all files in the folder.
FILTER_ENABLED = False  #@param {type:"boolean"}
FILTER_START_DATE = "2024-07-24"  #@param {type:"date"}
FILTER_START_TIME = "00:00"  #@param {type:"string"}
FILTER_END_DATE = "2025-07-25"  #@param {type:"date"}
FILTER_END_TIME = "23:59"  #@param {type:"string"}

#@markdown ---
#@markdown **Site coordinates** (optional) — latitude and longitude per recording site.
#@markdown Format: `SITE_NAME=lat,lon` separated by semicolons.
#@markdown The site name must match the subfolder name exactly (or the root folder name if no subfolders).
#@markdown Example: `POINT_A=-15.1,-47.2;POINT_B=-16.3,-48.1`
SITE_COORDINATES = ""  #@param {type:"string"}

FILENAME_PREFIX = FILENAME_PREFIX.strip()

_latlon_map = {}
for _e in SITE_COORDINATES.split(';'):
    _e = _e.strip()
    if '=' in _e:
        _n, _c = _e.split('=', 1)
        _p = _c.split(',')
        try: _latlon_map[_n.strip()] = (float(_p[0]), float(_p[1]))
        except Exception: pass

if not os.path.isdir(DRIVE_INPUT_DIR):
    print(f"WARNING: Folder not found: {DRIVE_INPUT_DIR}")
    print("Check the path above — make sure Google Drive is connected and the folder exists.")
else:
    print(f"Audio folder : {DRIVE_INPUT_DIR}")
    if FILENAME_PREFIX:
        print(f"Filename prefix : '{FILENAME_PREFIX}'  (expects files like {FILENAME_PREFIX}_YYYYMMDD_HHMMSS.wav)")
    else:
        print("Filename prefix : none  (expects files like YYYYMMDD_HHMMSS.wav)")
    if FILTER_ENABLED:
        print(f"Date filter     : {FILTER_START_DATE} {FILTER_START_TIME} → {FILTER_END_DATE} {FILTER_END_TIME}")
    else:
        print("Date filter     : disabled (all files will be analyzed)")
    if _latlon_map:
        for _sn, (_la, _lo) in _latlon_map.items():
            print(f"  Coordinates   : {_sn} → lat={_la}, lon={_lo}")
    else:
        print("Site coordinates: not set")

In [ ]:
#@title 🐦 BirdNET Settings { display-mode: "form" }

#@markdown **Recording site location** — used by BirdNET's geo model to filter species unlikely
#@markdown to occur at this location, reducing false positives. Use decimal degrees.
#@markdown If your sites are far apart, use the rough center point of your study area.
LATITUDE  = -27.0  #@param {type:"number"}
LONGITUDE = -48.0  #@param {type:"number"}

#@markdown ---
#@markdown **Geo filter score** — minimum probability for a species to be included in the
#@markdown location-based species list. Lower = more inclusive. Set to 0.0 to disable filtering
#@markdown and detect all ~6500 BirdNET species.
GEO_MIN_SCORE = 0.03  #@param {type:"slider", min:0.0, max:1.0, step:0.01}

#@markdown ---
#@markdown **Segment overlap (0.0–0.9)** — fraction of overlap between consecutive 3-second BirdNET chunks.
#@markdown 0.0 = no overlap (default). 0.5 = 50% overlap (1.5 s step, better coverage).
SEGMENT_OVERLAP = 0.0  #@param {type:"slider", min:0.0, max:0.9, step:0.1}

print(f"Location         : lat={LATITUDE}, lon={LONGITUDE}")
print(f"Geo filter score : {GEO_MIN_SCORE}{'  (disabled — all species)' if GEO_MIN_SCORE == 0.0 else ''}")
print(f"Segment overlap  : {SEGMENT_OVERLAP}")

---
## Step 3 — Scan audio files

This cell scans your audio folder, discovers all recording sites and files, and shows a summary of what will be analyzed.

**Site detection:**
- If your audio folder contains **subfolders**, each subfolder is treated as one recording site.
- If all audio files are **directly in the root folder**, the folder itself is treated as a single site.

**Datetime parsing:** timestamps are read from filenames using the prefix you configured. Files whose names do not match the expected pattern are still included — their date/time columns in results will be left blank.

In [ ]:
#@title 🔍 Scan { display-mode: "form" }

import os
import re
import glob as _glob
from collections import defaultdict
from datetime import datetime

def parse_file_datetime(filename, prefix=''):
    name = os.path.splitext(os.path.basename(filename))[0]
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
    if m:
        return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                        int(m.group(4)), int(m.group(5)), int(m.group(6)))
    if prefix:
        m = re.search(rf'{re.escape(prefix)}_(\d{{8}})_(\d{{6}})', name)
    else:
        m = re.search(r'(\d{8})_(\d{6})', name)
    if m:
        d, t = m.group(1), m.group(2)
        return datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                        int(t[:2]), int(t[2:4]), int(t[4:6]))
    return None

def get_site_name(audio_path, input_dir):
    parent = os.path.normpath(os.path.dirname(audio_path))
    root   = os.path.normpath(input_dir)
    if parent == root:
        return os.path.basename(root)
    return os.path.basename(parent)

if not os.path.isdir(DRIVE_INPUT_DIR):
    raise FileNotFoundError(
        f"Audio folder not found: {DRIVE_INPUT_DIR}\n"
        "Please check the path in the Audio Input form (Step 2)."
    )

all_audio = sorted(
    _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.wav'),  recursive=True) +
    _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.flac'), recursive=True) +
    _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.mp3'),  recursive=True)
)

if FILTER_ENABLED and all_audio:
    _start = datetime.strptime(f"{FILTER_START_DATE} {FILTER_START_TIME or '00:00'}", '%Y-%m-%d %H:%M')
    _end   = datetime.strptime(f"{FILTER_END_DATE} {FILTER_END_TIME or '23:59'}", '%Y-%m-%d %H:%M')
    _n_before = len(all_audio)
    all_audio = [
        f for f in all_audio
        if (lambda dt: dt is None or _start <= dt <= _end)(parse_file_datetime(f, FILENAME_PREFIX))
    ]
    print(f"Date filter  : {_start.strftime('%Y-%m-%d %H:%M')} → {_end.strftime('%Y-%m-%d %H:%M')}")
    print(f"Files excluded by filter : {_n_before - len(all_audio)}")
    print()

if not all_audio:
    print(f"No audio files found in: {DRIVE_INPUT_DIR}")
    print("Supported formats: .wav, .flac, .mp3")
else:
    sites = defaultdict(list)
    no_datetime = []

    for f in all_audio:
        site = get_site_name(f, DRIVE_INPUT_DIR)
        dt   = parse_file_datetime(f, FILENAME_PREFIX)
        sites[site].append((f, dt))
        if dt is None:
            no_datetime.append(os.path.basename(f))

    print(f"Audio folder : {DRIVE_INPUT_DIR}")
    print(f"Total files  : {len(all_audio)}")
    print(f"Sites found  : {len(sites)}")
    print()

    for site_name, entries in sorted(sites.items()):
        dts = [dt for _, dt in entries if dt is not None]
        if dts:
            date_range = f"{min(dts).strftime('%Y-%m-%d %H:%M')} → {max(dts).strftime('%Y-%m-%d %H:%M')}"
        else:
            date_range = '(no timestamps parsed)'
        print(f"  {site_name}")
        print(f"    Files : {len(entries)}")
        print(f"    Range : {date_range}")

    if no_datetime:
        print()
        print(f"  WARNING: {len(no_datetime)} file(s) did not match the filename pattern:")
        for fn in no_datetime[:5]:
            print(f"    {fn}")
        if len(no_datetime) > 5:
            print(f"    ... and {len(no_datetime) - 5} more")
        if FILENAME_PREFIX:
            print(f"  Expected pattern: {FILENAME_PREFIX}_YYYYMMDD_HHMMSS.*")
        else:
            print(f"  Expected pattern: YYYYMMDD_HHMMSS.*")
        print("  These files will still be analyzed — date/time columns in results will be blank.")

    print()
    print("Scan complete. Continue to Step 4.")

---
## Step 4 — Load BirdNET

This step loads two BirdNET components:

1. **Geo model** — scores how likely each of BirdNET's ~6 500 species is to occur at your location and time of year. Species below `GEO_MIN_SCORE` are excluded, which reduces false positives. Set `GEO_MIN_SCORE = 0.0` in BirdNET Settings to skip this filter and detect all species.
2. **Acoustic model** — the main neural network that listens to 3-second audio segments and scores each species.

> This cell may take a minute on the first run while BirdNET downloads its model weights.

In [ ]:
#@title 🐦 Load model { display-mode: "form" }

import birdnet
import tempfile
import datetime as _dt

MODEL_NAME = 'BirdNET_v2.4'

species_list_path = None

if GEO_MIN_SCORE > 0.0:
    # Use week from first parseable file datetime, fall back to current week.
    _first_dt = None
    for f in all_audio:
        _first_dt = parse_file_datetime(f, FILENAME_PREFIX)
        if _first_dt is not None:
            break
    week = min((_first_dt or _dt.datetime.today()).isocalendar()[1], 48)

    print(f'Running geo model: lat={LATITUDE}, lon={LONGITUDE}, week={week} ...')
    geo_model = birdnet.load('geo', '2.4', 'tf')
    geo_preds = geo_model.predict(LATITUDE, LONGITUDE, week=week)

    geo_df     = geo_preds.to_dataframe(sort_by='confidences')
    species_df = geo_df[geo_df['confidence'] >= GEO_MIN_SCORE]
    print(f'Species in location filter: {len(species_df)}')

    tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False)
    tmp.write('\n'.join(species_df['species_name'].tolist()))
    tmp.close()
    species_list_path = tmp.name
    print(f'Species list written to: {species_list_path}')
else:
    print('Location filtering disabled — all ~6500 BirdNET species will be considered.')

print()
print('Loading acoustic model ...')
acoustic_model = birdnet.load('acoustic', '2.4', 'tf')
print(f'BirdNET acoustic model ready.')
print(f'Model name : {MODEL_NAME}')

---
## Step 5 — Check already completed analyses

This cell scans your results folder to see which audio files have **already been analyzed**. Those files will be **skipped** in the next step, so you can safely re-run the analysis without repeating work you've already done.

In [ ]:
#@title 🔎 Check { display-mode: "form" }

import os
import re
import glob as _glob
from datetime import datetime

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

def parse_file_datetime(filename, prefix=''):
    name = os.path.splitext(os.path.basename(filename))[0]
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
    if m:
        return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                        int(m.group(4)), int(m.group(5)), int(m.group(6)))
    if prefix:
        m = re.search(rf'{re.escape(prefix)}_(\d{{8}})_(\d{{6}})', name)
    else:
        m = re.search(r'(\d{8})_(\d{6})', name)
    if m:
        d, t = m.group(1), m.group(2)
        return datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                        int(t[:2]), int(t[2:4]), int(t[4:6]))
    return None

def get_site_name(audio_path, input_dir):
    parent = os.path.normpath(os.path.dirname(audio_path))
    root   = os.path.normpath(input_dir)
    if parent == root:
        return os.path.basename(root)
    return os.path.basename(parent)

def get_result_path(audio_path, input_dir, results_dir, model_name, prefix=''):
    filename  = os.path.basename(audio_path)
    site_name = get_site_name(audio_path, input_dir)
    file_dt   = parse_file_datetime(filename, prefix)
    if file_dt is not None:
        dt_str    = file_dt.strftime('%Y%m%d_%H%M%S')
        result_fn = f"{site_name}_{dt_str}.{model_name}.results.csv"
    else:
        file_base = os.path.splitext(filename)[0]
        result_fn = f"{site_name}_{file_base}.{model_name}.results.csv"
    return os.path.join(results_dir, site_name, result_fn)

all_audio = sorted(
    _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.wav'),  recursive=True) +
    _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.flac'), recursive=True) +
    _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.mp3'),  recursive=True)
)

if FILTER_ENABLED and all_audio:
    _start = datetime.strptime(f"{FILTER_START_DATE} {FILTER_START_TIME or '00:00'}", '%Y-%m-%d %H:%M')
    _end   = datetime.strptime(f"{FILTER_END_DATE} {FILTER_END_TIME or '23:59'}", '%Y-%m-%d %H:%M')
    all_audio = [
        f for f in all_audio
        if (lambda dt: dt is None or _start <= dt <= _end)(parse_file_datetime(f, FILENAME_PREFIX))
    ]

if TIME_FILTER_ENABLED and all_audio:
    from datetime import time as _dtime
    _ps = TIME_FILTER_START.split(':')
    _pe = TIME_FILTER_END.split(':')
    _ts = _dtime(int(_ps[0]), int(_ps[1]))
    _te = _dtime(int(_pe[0]), int(_pe[1]))
    def _tod_ok(f):
        dt = parse_file_datetime(f, FILENAME_PREFIX)
        if dt is None:
            return True
        t = dt.time()
        return (_ts <= t <= _te) if _ts <= _te else (t >= _ts or t <= _te)
    all_audio = [f for f in all_audio if _tod_ok(f)]

to_process   = [f for f in all_audio
                if not os.path.exists(
                    get_result_path(f, DRIVE_INPUT_DIR, DRIVE_RESULTS_DIR,
                                    MODEL_NAME, FILENAME_PREFIX))]
already_done = [f for f in all_audio if f not in to_process]

print(f'Results folder : {DRIVE_RESULTS_DIR}')
print(f'Model name     : {MODEL_NAME}')
if FILTER_ENABLED:
    print(f'Date filter    : {FILTER_START_DATE} {FILTER_START_TIME} → {FILTER_END_DATE} {FILTER_END_TIME}')
print()
print(f'Total audio files  : {len(all_audio)}')
print(f'Already analyzed   : {len(already_done)}')
print(f'Remaining to run   : {len(to_process)}')

if already_done:
    print()
    print('Already done (will be skipped):')
    for f in already_done:
        rp = get_result_path(f, DRIVE_INPUT_DIR, DRIVE_RESULTS_DIR,
                             MODEL_NAME, FILENAME_PREFIX)
        print(f"  {os.path.basename(f)}  →  {os.path.basename(rp)}")

if not to_process:
    print()
    print('All files have already been analyzed. Nothing to do!')
    print('If you want to re-analyze, delete the corresponding results files from your Drive first.')
else:
    print()
    print(f'Ready to analyze {len(to_process)} file(s). Proceed to Step 6.')

---
## Step 6 — Run the analysis and save results

This is the main analysis step. For each audio file, BirdNET splits the recording into 3-second segments, scores each species, and saves detections above your confidence threshold.

**Result files** are saved as:
```
DRIVE_RESULTS_DIR / SITE_NAME / audio_file.BirdNET_v2.4.results.csv
```

Each results file has columns: `site`, `date`, `time`, `start_time`, `end_time`, `scientific_name`, `common_name`, `confidence`

- `site` — the recording site name (subfolder name, or root folder name if no subfolders).
- `date` / `time` — timestamp of the detection segment, read from the filename.
- `start_time` / `end_time` — offset in seconds from the beginning of the audio file.

> Depending on the number of files and the length of recordings, this step can take a while. Progress is shown below.

In [ ]:
#@title 🚀 Run { display-mode: "form" }

import csv
import os
import re
import shutil
import time
from datetime import datetime, timedelta

def parse_file_datetime(filename, prefix=''):
    name = os.path.splitext(os.path.basename(filename))[0]
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
    if m:
        return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                        int(m.group(4)), int(m.group(5)), int(m.group(6)))
    if prefix:
        m = re.search(rf'{re.escape(prefix)}_(\d{{8}})_(\d{{6}})', name)
    else:
        m = re.search(r'(\d{8})_(\d{6})', name)
    if m:
        d, t = m.group(1), m.group(2)
        return datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                        int(t[:2]), int(t[2:4]), int(t[4:6]))
    return None

def get_site_name(audio_path, input_dir):
    parent = os.path.normpath(os.path.dirname(audio_path))
    root   = os.path.normpath(input_dir)
    if parent == root:
        return os.path.basename(root)
    return os.path.basename(parent)

def _time_to_seconds(t):
    if hasattr(t, 'total_seconds'):
        return t.total_seconds()
    s = str(t)
    if ':' in s:
        parts = s.split(':')
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    return float(s)


def _in_time_window(file_dt, start_str, end_str):
    from datetime import time as _dtime
    if file_dt is None:
        return True
    t = file_dt.time()
    ps, pe = start_str.split(':'), end_str.split(':')
    t_start = _dtime(int(ps[0]), int(ps[1]))
    t_end   = _dtime(int(pe[0]), int(pe[1]))
    if t_start <= t_end:
        return t_start <= t <= t_end
    return t >= t_start or t <= t_end

total_detections = 0
total_start      = time.time()

if not to_process:
    print('No files to process. All analyses are already complete.')
else:
    print(f'Starting BirdNET analysis of {len(to_process)} file(s)')
    print(f'Confidence threshold : {MIN_CONFIDENCE}')
    print(f'Geo filter score     : {GEO_MIN_SCORE}{"  (disabled)" if GEO_MIN_SCORE == 0.0 else ""}')
    if FILTER_TYPE != 'none' or AUDIO_SPEED != 1.0:
        _pp = []
        if FILTER_TYPE != 'none': _pp.append(f'filter={FILTER_TYPE}')
        if AUDIO_SPEED != 1.0:   _pp.append(f'speed={AUDIO_SPEED}x')
        print(f'Preprocessing        : {", ".join(_pp)}')
    print('=' * 65)

    for file_idx, audio_path in enumerate(to_process, 1):
        filename  = os.path.basename(audio_path)
        site_name = get_site_name(audio_path, DRIVE_INPUT_DIR)
        file_dt   = parse_file_datetime(filename, FILENAME_PREFIX)
        if TIME_FILTER_ENABLED and not _in_time_window(file_dt, TIME_FILTER_START, TIME_FILTER_END):
            print(f'[{file_idx}/{len(to_process)}] {filename}  — skipped (outside {TIME_FILTER_START}–{TIME_FILTER_END})')
            continue

        if file_dt is not None:
            dt_str    = file_dt.strftime('%Y%m%d_%H%M%S')
            result_fn = f"{site_name}_{dt_str}.{MODEL_NAME}.results.csv"
        else:
            file_base = os.path.splitext(filename)[0]
            result_fn = f"{site_name}_{file_base}.{MODEL_NAME}.results.csv"
            print(f"  WARNING: could not parse datetime from '{filename}' — using filename as fallback.")

        print(f'[{file_idx}/{len(to_process)}] {filename}  (site: {site_name})')

        file_start = time.time()
        rows = []

        os.makedirs('/content/audio_tmp', exist_ok=True)
        _local = os.path.join('/content/audio_tmp', os.path.basename(audio_path))
        shutil.copy2(audio_path, _local)
        try:
            kwargs = {}
            if species_list_path:
                kwargs['custom_species_list'] = species_list_path
            if SEGMENT_OVERLAP > 0.0:
                kwargs['overlap_duration_s'] = round(SEGMENT_OVERLAP * 3.0, 1)
            if FILTER_TYPE == 'lowpass':
                kwargs['bandpass_fmax'] = FILTER_HIGH_HZ
            elif FILTER_TYPE == 'highpass':
                kwargs['bandpass_fmin'] = FILTER_LOW_HZ
            elif FILTER_TYPE == 'bandpass':
                kwargs['bandpass_fmin'] = FILTER_LOW_HZ
                kwargs['bandpass_fmax'] = FILTER_HIGH_HZ
            if AUDIO_SPEED != 1.0:
                kwargs['speed'] = AUDIO_SPEED
            result = acoustic_model.predict(_local, **kwargs)
            preds  = result.to_dataframe()
        except Exception as e:
            os.remove(_local)
            print(f'  ERROR during analysis: {e} — skipping.')
            continue
        os.remove(_local)

        if not preds.empty:
            preds = preds[preds['confidence'] >= MIN_CONFIDENCE].copy()
            for _, row in preds.iterrows():
                start_s = _time_to_seconds(row['start_time'])
                end_s   = _time_to_seconds(row['end_time'])

                sp = str(row['species_name']).split('_', 1)
                scientific = sp[0].strip()
                common     = sp[1].strip() if len(sp) > 1 else ''

                if file_dt is not None:
                    date_str = file_dt.strftime('%Y-%m-%d')
                    time_str = file_dt.strftime('%H:%M:%S')
                else:
                    date_str = ''
                    time_str = ''

                site_lat, site_lon = _latlon_map.get(site_name, ('', ''))
                rows.append([
                    site_name, site_lat, site_lon, date_str, time_str,
                    f'{start_s:.1f}', f'{end_s:.1f}',
                    scientific, common, f'{float(row["confidence"]):.4f}'
                ])

        elapsed = time.time() - file_start

        site_dir    = os.path.join(DRIVE_RESULTS_DIR, site_name)
        os.makedirs(site_dir, exist_ok=True)
        result_path = os.path.join(site_dir, result_fn)

        with open(result_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['site', 'lat', 'lon', 'date', 'time', 'start_time', 'end_time',
                             'scientific_name', 'common_name', 'confidence'])
            writer.writerows(rows)

        total_detections += len(rows)
        print(f'  -> {len(rows)} detection(s)  |  {elapsed:.1f}s  →  {result_fn}')

    total_elapsed = time.time() - total_start
    print()
    print('=' * 65)
    print(f'Analysis complete!')
    print(f'  Files analyzed   : {len(to_process)}')
    print(f'  Total detections : {total_detections}')
    print(f'  Total time       : {total_elapsed:.1f}s')
    print(f'  Results saved to : {DRIVE_RESULTS_DIR}')